# Enums and Dataclasses

## Enums
An **Enum** is a set of named, constant values. Use them instead of magic strings or integers.

## Dataclasses
A **dataclass** is a class mainly used to store data. The `@dataclass` decorator auto-generates `__init__`, `__repr__`, `__eq__` for you.

Both tools make your code more **readable**, **safe**, and **self-documenting**.

## Part 1: Enums

### 1.1 Basic Enum

In [ ]:
from enum import Enum, auto

class Direction(Enum):
    NORTH = 'north'
    SOUTH = 'south'
    EAST = 'east'
    WEST = 'west'

# Access members
d = Direction.NORTH
print(d)           # Direction.NORTH
print(d.name)      # 'NORTH'
print(d.value)     # 'north'

# Comparison
print(d == Direction.NORTH)  # True
print(d == Direction.SOUTH)  # False

# Iteration
print('All directions:')
for direction in Direction:
    print(f'  {direction.name}: {direction.value}')

In [ ]:
# auto() — assign values automatically
class Color(Enum):
    RED = auto()    # 1
    GREEN = auto()  # 2
    BLUE = auto()   # 3

for c in Color:
    print(f'{c.name} = {c.value}')

# Look up by value
c = Color(2)
print(c)  # Color.GREEN

# Look up by name
c2 = Color['BLUE']
print(c2)  # Color.BLUE

In [ ]:
# IntEnum — enum that behaves like an int
from enum import IntEnum

class Priority(IntEnum):
    LOW = 1
    MEDIUM = 2
    HIGH = 3
    CRITICAL = 4

# IntEnum can be compared with integers
p = Priority.HIGH
print(p > 2)           # True
print(p == 3)          # True
print(p + 1)           # 4

# Useful for sorting
tasks = [
    ('Fix bug', Priority.CRITICAL),
    ('Write docs', Priority.LOW),
    ('Review PR', Priority.HIGH),
]
tasks.sort(key=lambda t: t[1], reverse=True)
for task, priority in tasks:
    print(f'[{priority.name}] {task}')

In [ ]:
# Enum with methods
from enum import Enum

class Status(Enum):
    PENDING = 'pending'
    ACTIVE = 'active'
    COMPLETED = 'completed'
    CANCELLED = 'cancelled'
    
    def is_terminal(self):
        return self in (Status.COMPLETED, Status.CANCELLED)
    
    def can_transition_to(self, new_status):
        transitions = {
            Status.PENDING: {Status.ACTIVE, Status.CANCELLED},
            Status.ACTIVE: {Status.COMPLETED, Status.CANCELLED},
        }
        allowed = transitions.get(self, set())
        return new_status in allowed

s = Status.PENDING
print(f'Is terminal: {s.is_terminal()}')
print(f'Can go to ACTIVE: {s.can_transition_to(Status.ACTIVE)}')
print(f'Can go to COMPLETED: {s.can_transition_to(Status.COMPLETED)}')

## Part 2: Dataclasses

### 2.1 Basic Dataclass

In [ ]:
from dataclasses import dataclass

@dataclass
class Point:
    x: float
    y: float

# __init__, __repr__, __eq__ are auto-generated
p1 = Point(1.0, 2.0)
p2 = Point(3.0, 4.0)
p3 = Point(1.0, 2.0)

print(p1)          # Point(x=1.0, y=2.0)
print(p1 == p3)    # True — compares field values
print(p1 == p2)    # False

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime

@dataclass
class Employee:
    name: str
    department: str
    salary: float
    id: int = field(default=0)
    skills: list[str] = field(default_factory=list)  # mutable default!
    hired_at: datetime = field(default_factory=datetime.now)
    
    def annual_salary(self) -> float:
        return self.salary * 12

e1 = Employee('Alice', 'Engineering', 5000, id=1, skills=['Python', 'SQL'])
e2 = Employee('Bob', 'Marketing', 4500, id=2)

print(e1)
print(e1.annual_salary())
print(e2.skills)  # [] — independent list

### 2.2 `__post_init__` — Validation After Initialization

In [ ]:
from dataclasses import dataclass

@dataclass
class Circle:
    radius: float
    
    def __post_init__(self):
        if self.radius < 0:
            raise ValueError(f'Radius must be non-negative, got {self.radius}')
    
    @property
    def area(self):
        import math
        return math.pi * self.radius ** 2
    
    @property
    def circumference(self):
        import math
        return 2 * math.pi * self.radius

c = Circle(5.0)
print(f'Area: {c.area:.2f}')
print(f'Circumference: {c.circumference:.2f}')

try:
    bad = Circle(-1)
except ValueError as e:
    print(f'Error: {e}')

### 2.3 Frozen Dataclasses — Immutable

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)  # immutable — like a named tuple
class Coordinate:
    latitude: float
    longitude: float

loc = Coordinate(51.5, -0.1)  # London
print(loc)

try:
    loc.latitude = 48.8  # raises FrozenInstanceError
except Exception as e:
    print(f'Error: {e}')

# Frozen dataclasses are hashable — can be used in sets/dict keys
locations = {Coordinate(51.5, -0.1), Coordinate(48.8, 2.3)}
print(len(locations))

### 2.4 Dataclass vs NamedTuple vs Dict

| Feature | dict | namedtuple | dataclass |
|---------|------|------------|----------|
| Mutable | Yes | No | Yes (frozen=True for No) |
| Methods | No | Limited | Yes |
| Type hints | No | Optional | Yes |
| Default values | N/A | Limited | Yes |
| Readable repr | No | Yes | Yes |
| Performance | Fast | Fastest | Fast |

In [ ]:
from dataclasses import dataclass
from collections import namedtuple

# As a dict
user_dict = {'name': 'Alice', 'age': 30}

# As a namedtuple
UserTuple = namedtuple('User', ['name', 'age'])
user_tuple = UserTuple('Alice', 30)

# As a dataclass (PREFERRED for complex data)
@dataclass
class User:
    name: str
    age: int
    
    def greet(self):
        return f'Hi, I am {self.name}!'

user_dc = User('Alice', 30)

print(user_dc)
print(user_dc.greet())
print(user_dc.name)

## Common Mistakes

### Mistake 1: Comparing Enums with their Values Directly

In [ ]:
from enum import Enum

class Status(Enum):
    ACTIVE = 'active'

s = Status.ACTIVE

# BAD — comparing with the value
if s == 'active':  # False! Enum members != their values
    print('This will NOT print')

# GOOD — compare with the enum member
if s == Status.ACTIVE:
    print('This will print')

# Or compare value explicitly
if s.value == 'active':
    print('This also works')

### Mistake 2: Mutable Default in Dataclass Without `field()`

In [ ]:
from dataclasses import dataclass, field

# BAD — will raise ValueError at class definition time
# @dataclass
# class BadClass:
#     items: list = []  # ValueError: mutable default not allowed!

# GOOD
@dataclass
class GoodClass:
    items: list = field(default_factory=list)

a = GoodClass()
b = GoodClass()
a.items.append(1)
print('a:', a.items)  # [1]
print('b:', b.items)  # [] — independent!

## Mini-Project: Order Management System

In [ ]:
from dataclasses import dataclass, field
from enum import Enum, auto
from datetime import datetime


class OrderStatus(Enum):
    PENDING = auto()
    CONFIRMED = auto()
    SHIPPED = auto()
    DELIVERED = auto()
    CANCELLED = auto()
    
    def next_status(self):
        transitions = {
            OrderStatus.PENDING: OrderStatus.CONFIRMED,
            OrderStatus.CONFIRMED: OrderStatus.SHIPPED,
            OrderStatus.SHIPPED: OrderStatus.DELIVERED,
        }
        return transitions.get(self)


class PaymentMethod(Enum):
    CREDIT_CARD = 'Credit Card'
    PAYPAL = 'PayPal'
    BANK_TRANSFER = 'Bank Transfer'


@dataclass
class OrderItem:
    name: str
    quantity: int
    unit_price: float
    
    @property
    def total(self) -> float:
        return self.quantity * self.unit_price


@dataclass
class Order:
    order_id: str
    customer: str
    payment_method: PaymentMethod
    items: list[OrderItem] = field(default_factory=list)
    status: OrderStatus = OrderStatus.PENDING
    created_at: datetime = field(default_factory=datetime.now)
    status_history: list[tuple] = field(default_factory=list)
    
    def add_item(self, name: str, quantity: int, price: float):
        self.items.append(OrderItem(name, quantity, price))
    
    def subtotal(self) -> float:
        return sum(item.total for item in self.items)
    
    def advance_status(self):
        next_s = self.status.next_status()
        if next_s is None:
            print(f'Order is already in final state: {self.status.name}')
            return
        old = self.status
        self.status = next_s
        self.status_history.append((datetime.now(), old, next_s))
        print(f'Order {self.order_id}: {old.name} → {next_s.name}')
    
    def cancel(self):
        if self.status == OrderStatus.DELIVERED:
            raise ValueError('Cannot cancel a delivered order')
        self.status = OrderStatus.CANCELLED
        print(f'Order {self.order_id} cancelled')
    
    def receipt(self) -> str:
        lines = [f'Order #{self.order_id} | {self.customer} | {self.payment_method.value}']
        lines.append(f'Status: {self.status.name}')
        lines.append('-' * 40)
        for item in self.items:
            lines.append(f'  {item.name:20} x{item.quantity} @ £{item.unit_price:.2f} = £{item.total:.2f}')
        lines.append(f'  {"TOTAL":35} £{self.subtotal():.2f}')
        return '\n'.join(lines)


# Demo
order = Order('ORD-001', 'Alice Smith', PaymentMethod.CREDIT_CARD)
order.add_item('Laptop', 1, 999.99)
order.add_item('Mouse', 2, 29.99)
order.add_item('USB Hub', 1, 49.99)

print(order.receipt())
print()

# Advance through states
order.advance_status()  # PENDING → CONFIRMED
order.advance_status()  # CONFIRMED → SHIPPED
order.advance_status()  # SHIPPED → DELIVERED
order.advance_status()  # already final